# Reproduce the full simulation study from scratch

The slow, complete path: every method on every DGP at the headline n, the trajectory, the poly baselines, the recovery log and the reproducibility check. Several hours, dominated by the neural-net and PySR fits.

For routine work after the first run, use **run_remaining.ipynb** instead.


## Setup


In [ ]:
import pickle
import numpy as np, pandas as pd

import config as C
from dgp import DGP_REGISTRY
from estimators import SR_PDS_LOG
from simulation import run_all, seed_robustness_check
from evaluate import evaluate_all, evaluate_recovery, print_summary
from pipeline import build_registry, run_trajectory
from plots import plot_n_trajectory

registry = build_registry(include_poly=True, improved_cf=True)
dgp_keys = list(DGP_REGISTRY.keys())
est_keys = list(registry.keys())
print('DGPs:', dgp_keys)
print('methods:', est_keys)


## 1 — Headline fixed-n grid

Every method on every DGP at n = 500. The cross-fit variant runs at a reduced replication count.


In [ ]:
SR_PDS_LOG.clear()
headline = run_all(DGP_REGISTRY, registry,
                   dgp_keys=dgp_keys, estimator_keys=est_keys,
                   n=C.N_HEADLINE, p=C.P, s=C.S, beta0=C.BETA0,
                   n_reps=C.N_REPS_MAIN, n_jobs=C.N_JOBS,
                   save_dir=str(C.RESULTS_DIR),
                   reps_overrides={'sr_pds_cf': C.N_REPS_CF})
headline.to_pickle(C.RESULTS_DIR/'headline.pkl')
with open(C.RESULTS_DIR/'sr_pds_log_headline.pkl','wb') as f:
    pickle.dump(list(SR_PDS_LOG), f)

metrics = evaluate_all(headline, C.BETA0)
metrics.to_csv(C.RESULTS_DIR/'headline_metrics.csv', index=False)
print_summary(metrics, C.BETA0)


## 2 — Term recovery


In [ ]:
recovery = evaluate_recovery(list(SR_PDS_LOG), DGP_REGISTRY)
recovery.to_csv(C.RESULTS_DIR/'recovery.csv', index=False)
print('recovery records:', len(recovery))
recovery.head(20)


## 3 — Sample-size trajectory

Confounding designs across the n grid. The cross-fit variant is excluded (reported at the headline n only). Saves after each cell; resumes if interrupted.


In [ ]:
traj_keys = [k for k in est_keys if k != 'sr_pds_cf']
SR_PDS_LOG.clear()
traj = run_trajectory(DGP_REGISTRY, registry, traj_keys, C.CONF_DGPS,
                      n_grid=C.N_GRID_TRAJ, n_reps=C.N_REPS_TRAJ,
                      beta0=C.BETA0, p=C.P, s=C.S, n_jobs=C.N_JOBS,
                      save_path=C.RESULTS_DIR/'trajectory.pkl')
traj.to_pickle(C.RESULTS_DIR/'trajectory.pkl')
print('trajectory rows:', len(traj))


## 4 — Reproducibility check


In [ ]:
seed_df = seed_robustness_check(DGP_REGISTRY['dgp6']['fn'], dgp_key='dgp6',
                                n=C.N_HEADLINE, p=C.P, s=C.S, beta0=C.BETA0,
                                pysr_seeds=list(range(1, C.N_REPS_SEEDS+1)),
                                save_dir=str(C.RESULTS_DIR))
seed_df


## 5 — Trajectory figure


In [ ]:
plot_n_trajectory(traj, beta0=C.BETA0, save=True)
print('done -> figures/n_trajectory.png')
